In [11]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/juvecom


In [6]:
import re
from datetime import timedelta
from faker import Faker
from database.conexao import db, conectar

fake = Faker("pt_BR")
Faker.seed(2026)

TOTAL_CLIENTES = 20

def numeros(texto):
    return re.sub(r"\D", "", texto)

def conferir_vazia(tabela):
    total = db.execute_sql(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    if total:
        raise RuntimeError(f"{tabela} já possui {total} registro(s).")

### tb_pessoa

In [3]:
import json
import random
from faker import Faker
from database.conexao import db, conectar

fake = Faker("pt_BR")
Faker.seed(2026)
random.seed(2026)


def gerar_cnpj():
    numeros = [random.randint(0, 9) for _ in range(12)]

    for pesos in (
        [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2],
        [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2],
    ):
        resto = sum(n * peso for n, peso in zip(numeros, pesos)) % 11
        numeros.append(0 if resto < 2 else 11 - resto)

    return "".join(map(str, numeros))


pessoas = []

for tipo in ["F"] * 10 + ["J"] * 10:
    dados = {
        "endereco": {
            "logradouro": fake.street_name(),
            "numero": fake.building_number(),
            "cidade": fake.city(),
            "uf": fake.state_abbr(),
            "cep": fake.postcode()
        }
    }

    pessoas.append((
        tipo,
        fake.name() if tipo == "F" else fake.company()[:150],
        fake.ssn() if tipo == "F" else gerar_cnpj(),
        fake.phone_number()[:20],
        fake.email(),
        json.dumps(dados, ensure_ascii=False)
    ))

with conectar():
    with db.atomic():
        for pessoa in pessoas:
            db.execute_sql(
                """
                INSERT INTO tb_pessoa
                    (tp_pessoa, nm_pessoa, cpf_cnpj, telefone, email, dados)
                VALUES (%s, %s, %s, %s, %s, %s::jsonb)
                ON CONFLICT (cpf_cnpj) DO NOTHING
                """,
                pessoa
            )

print("Carga concluída.")

with conectar():
    registros = db.execute_sql(
        "SELECT cd_pessoa, tp_pessoa, nm_pessoa, cpf_cnpj FROM tb_pessoa ORDER BY cd_pessoa"
    ).fetchall()

for registro in registros:
    print(registro)

Carga concluída.
(1, 'F', 'Rael Albuquerque', '75392841023')
(2, 'F', 'Vicente Rios', '16793802496')
(3, 'F', 'Beatriz Macedo', '84713502626')
(4, 'F', 'Caroline Peixoto', '84135792600')
(5, 'F', 'Stephany Costela', '65741039243')
(6, 'F', 'Luan Moreira', '79146203516')
(7, 'F', 'Ana Liz Teixeira', '52148907611')
(8, 'F', 'Maria Fernanda Montenegro', '46982037169')
(9, 'F', 'Maria Flor da Costa', '63214089560')
(10, 'F', 'Kamilly Fernandes', '35897642109')
(11, 'J', 'Barros S.A.', '15881399869882')
(12, 'J', 'Freitas S/A', '79730911417033')
(13, 'J', 'Azevedo S/A', '75364556815127')
(14, 'J', 'Macedo', '88447294055713')
(15, 'J', 'Albuquerque da Mata - EI', '61698716897620')
(16, 'J', 'Martins Costela - EI', '84697880932030')
(17, 'J', 'Fonseca', '81967273177893')
(18, 'J', 'Moura', '48648876634425')
(19, 'J', 'da Mota Ltda.', '00297875347717')
(20, 'J', 'Araújo S/A', '87680457178400')


### tb_loja

In [4]:
import random
from faker import Faker
from database.conexao import db, conectar

fake = Faker("pt_BR")
random.seed(27)

with conectar():
    pessoas = db.execute_sql("""
        SELECT p.cd_pessoa
        FROM tb_pessoa p
        WHERE p.cd_pessoa <> 0 AND p.fl_ativo
          AND NOT EXISTS (
              SELECT 1 FROM tb_contrato c WHERE c.cd_pessoa = p.cd_pessoa
          )
        ORDER BY p.cd_pessoa
    """).fetchall()

    with db.atomic():
        for (cd_pessoa,) in pessoas:
            existentes = db.execute_sql(
                "SELECT count(*) FROM tb_loja WHERE cd_pessoa = %s", (cd_pessoa,)
            ).fetchone()[0]

            for numero in range(existentes + 1, random.randint(1, 3) + 1):
                shop_id = 90_000_000_000 + cd_pessoa * 10 + numero
                db.execute_sql("""
                    INSERT INTO tb_loja (cd_pessoa, nm_loja, shop_id, url_loja)
                    VALUES (%s, %s, %s, %s)
                """, (
                    cd_pessoa,
                    f"{fake.word().title()} Store",
                    shop_id,
                    f"https://shopee.com.br/shop/{shop_id}"
                ))

print(f"Pessoas disponíveis para simulação: {len(pessoas)}")

Pessoas disponíveis para simulação: 20


### tb_contrato

In [7]:
from datetime import timedelta
from decimal import Decimal
from database.conexao import db, conectar

lojas_por_contrato = {}

with conectar():
    with db.atomic():
        conferir_vazia("tb_contrato")

        inicio = db.execute_sql("SELECT CURRENT_DATE").fetchone()[0]

        pessoas = db.execute_sql("""
            SELECT p.cd_pessoa
            FROM tb_pessoa p
            WHERE p.cd_pessoa <> 0 AND p.fl_ativo
              AND EXISTS (
                  SELECT 1 FROM tb_loja l
                  WHERE l.cd_pessoa = p.cd_pessoa AND l.fl_ativo
              )
            ORDER BY p.cd_pessoa
        """).fetchall()

        planos = db.execute_sql("""
            SELECT nm_plano, ds_plano, periodicidade_dias, valor
            FROM tb_plano
            WHERE fl_ativo
            ORDER BY cd_plano
        """).fetchall()

        status = db.execute_sql("""
            SELECT cd_status
            FROM tb_contrato_status
            WHERE nm_status = 'ativo'
        """).fetchone()

        if not pessoas or not planos or status is None:
            raise RuntimeError("Cadastre pessoas, lojas, planos e o status ativo antes dos contratos.")

        for indice, (cd_pessoa,) in enumerate(pessoas):
            lojas = db.execute_sql("""
                SELECT cd_loja FROM tb_loja
                WHERE cd_pessoa = %s AND fl_ativo
                ORDER BY cd_loja
            """, (cd_pessoa,)).fetchall()

            ids_lojas = [cd_loja for (cd_loja,) in lojas]
            quantidade = len(ids_lojas)
            nm_plano, ds_plano, dias, valor = planos[indice % len(planos)]

            faixa = db.execute_sql("""
                SELECT desconto
                FROM tb_desconto_loja
                WHERE qt_loja @> %s::integer
                  AND vigencia @> %s::date
            """, (quantidade, inicio)).fetchone()

            if faixa is None:
                raise RuntimeError(f"Sem desconto cadastrado para {quantidade} lojas em {inicio}")

            parcelas = 1 if dias <= 30 else dias // 30
            vl_bruto = Decimal(valor) * quantidade * parcelas
            vl_desconto_lojas = vl_bruto * Decimal(faixa[0]) / 100
            vl_desconto_manual = Decimal("0.00")
            vl_acrescimo = Decimal("0.00")
            vl_contrato = vl_bruto - vl_desconto_lojas - vl_desconto_manual + vl_acrescimo
            fim = inicio + timedelta(days=dias)

            cd_contrato = db.execute_sql("""
                INSERT INTO tb_contrato
                    (cd_pessoa, cd_status, nm_plano, ds_plano, vigencia,
                     vl_bruto, vl_desconto_lojas, vl_desconto_manual,
                     vl_acrescimo, vl_contrato, ds_contrato)
                VALUES
                    (%s, %s, %s, %s, daterange(%s::date, %s::date, '[)'),
                     %s, %s, %s, %s, %s, %s)
                RETURNING cd_contrato
            """, (
                cd_pessoa, status[0], nm_plano, ds_plano, inicio, fim,
                vl_bruto, vl_desconto_lojas, vl_desconto_manual,
                vl_acrescimo, vl_contrato, "Contrato de demonstração"
            )).fetchone()[0]

            lojas_por_contrato[cd_contrato] = (cd_pessoa, ids_lojas)

    print("Contratos:", db.execute_sql("SELECT COUNT(*) FROM tb_contrato").fetchone()[0])

Contratos: 20


### tb_contrato_loja

In [8]:
from database.conexao import db, conectar

if "lojas_por_contrato" not in globals() or not lojas_por_contrato:
    raise RuntimeError(
        "lojas_por_contrato não está disponível. "
        "Se o kernel foi reiniciado, não repita a carga de contratos; "
        "recupere os vínculos a partir do banco."
    )

inseridos = 0

with conectar():
    with db.atomic():
        for cd_contrato, (cd_pessoa, ids_lojas) in lojas_por_contrato.items():
            for cd_loja in ids_lojas:
                cursor = db.execute_sql("""
                    INSERT INTO tb_contrato_loja
                        (cd_contrato, cd_loja, cd_pessoa)
                    VALUES (%s, %s, %s)
                    ON CONFLICT (cd_contrato, cd_loja) DO NOTHING
                """, (cd_contrato, cd_loja, cd_pessoa))

                inseridos += cursor.rowcount

    total = db.execute_sql(
        "SELECT COUNT(*) FROM tb_contrato_loja"
    ).fetchone()[0]

print(f"Vínculos inseridos agora: {inseridos}")
print(f"Total em tb_contrato_loja: {total}")

Vínculos inseridos agora: 37
Total em tb_contrato_loja: 37


### Parcelas

In [10]:
from database.conexao import db, conectar

with conectar():
    with db.atomic():
        conferir_vazia("tb_parcelas")

        cursor = db.execute_sql("""
            WITH contratos AS (
                SELECT c.*,
                       CASE
                           WHEN upper(c.vigencia) - lower(c.vigencia) <= 30 THEN 1
                           ELSE (upper(c.vigencia) - lower(c.vigencia)) / 30
                       END AS qt_parcelas
                FROM tb_contrato c
            ),
            distribuicao AS (
                SELECT c.*,
                       n.nr_parcela,
                       c.vl_bruto + c.vl_acrescimo AS total_parcelas_bruto,
                       c.vl_desconto_lojas + c.vl_desconto_manual AS total_descontos,
                       ROUND((c.vl_bruto + c.vl_acrescimo) / c.qt_parcelas, 2) AS bruto_regular,
                       ROUND((c.vl_desconto_lojas + c.vl_desconto_manual) / c.qt_parcelas, 2) AS desconto_regular
                FROM contratos c
                CROSS JOIN LATERAL generate_series(1, c.qt_parcelas) AS n(nr_parcela)
            )
            INSERT INTO tb_parcelas
                (cd_contrato, nr_parcela, dt_vencimento, vl_parcela, vl_desconto)
            SELECT cd_contrato,
                   nr_parcela,
                   CASE
                       WHEN qt_parcelas = 1 THEN upper(vigencia) - 1
                       ELSE LEAST(
                           (lower(vigencia) + (nr_parcela || ' months')::interval)::date,
                           upper(vigencia) - 1
                       )
                   END AS dt_vencimento,
                   CASE
                       WHEN nr_parcela = qt_parcelas
                           THEN total_parcelas_bruto - bruto_regular * (qt_parcelas - 1)
                       ELSE bruto_regular
                   END AS vl_parcela,
                   CASE
                       WHEN nr_parcela = qt_parcelas
                           THEN total_descontos - desconto_regular * (qt_parcelas - 1)
                       ELSE desconto_regular
                   END AS vl_desconto
            FROM distribuicao
            ORDER BY cd_contrato, nr_parcela
        """)

    print(f"Parcelas inseridas: {cursor.rowcount}")

Parcelas inseridas: 110


### Carga Total

In [ ]:
import re
from datetime import timedelta
from faker import Faker
from database.conexao import db, conectar

fake = Faker("pt_BR")
Faker.seed(2026)

TOTAL_CLIENTES = 20


def somente_numeros(texto):
    return re.sub(r"\D", "", texto)


with conectar():
    with db.atomic():
        tabelas = ("tb_cliente", "tb_loja", "tb_contrato", "tb_plano_contrato")

        for tabela in tabelas:
            total = db.execute_sql(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
            if total:
                raise RuntimeError(f"{tabela} já possui {total} registro(s). Carga interrompida.")

        planos = db.execute_sql("""
            SELECT cd_plano, periodicidade_dias, valor
            FROM tb_plano
            WHERE fl_ativo = true
            ORDER BY cd_plano
        """).fetchall()

        if not planos:
            raise RuntimeError("Cadastre os planos antes de executar esta carga.")

        for numero in range(1, TOTAL_CLIENTES + 1):
            pessoa_juridica = numero % 2 == 0
            nome = fake.company() if pessoa_juridica else fake.name()
            cpf_cnpj = somente_numeros(fake.cnpj() if pessoa_juridica else fake.cpf())

            cd_cliente = db.execute_sql("""
                INSERT INTO tb_cliente (nm_cliente, cpf_cnpj, telefone, email)
                VALUES (%s, %s, %s, %s)
                RETURNING cd_cliente
            """, (nome, cpf_cnpj, fake.phone_number()[:20], fake.email())).fetchone()[0]

            db.execute_sql("""
                INSERT INTO tb_loja (cd_cliente, nm_loja)
                VALUES (%s, %s)
            """, (cd_cliente, f"Loja {fake.word().title()} {numero:03d}"))

            cd_plano, periodicidade_dias, valor = planos[(numero - 1) % len(planos)]
            dt_inicio = fake.date_between(start_date="-180d", end_date="today")
            dt_fim = dt_inicio + timedelta(days=periodicidade_dias)

            cd_contrato = db.execute_sql("""
                INSERT INTO tb_contrato (cd_cliente, dt_inicio, dt_fim, vl_contrato, ds_contrato)
                VALUES (%s, %s, %s, %s, %s)
                RETURNING cd_contrato
            """, (cd_cliente, dt_inicio, dt_fim, valor, "Contrato de demonstração")).fetchone()[0]

            db.execute_sql("""
                INSERT INTO tb_plano_contrato (cd_contrato, cd_plano, vl_plano)
                VALUES (%s, %s, %s)
            """, (cd_contrato, cd_plano, valor))

print(f"Carga concluída: {TOTAL_CLIENTES} clientes, lojas e contratos.")